# Sandbox alumno 1

Carga de datos directamente desde Hugging Face buckets.

In [ ]:
import re
import pandas as pd
from huggingface_hub import HfFileSystem

BASE_BUCKET = "hf://buckets/Cesar77RR/openstack-aiops-project"
DATASET_ROOT = f"{BASE_BUCKET}/Fault-Injection-Dataset"
FS = HfFileSystem()

In [ ]:
# Metadata del componente Nova desde Hugging Face
nova_tsv = f"{DATASET_ROOT}/nova.tsv"
with FS.open(nova_tsv, "rt", encoding="utf-8") as f:
    df_meta = pd.read_csv(f, sep='\t')

print("Dimensiones de la metadata:", df_meta.shape)
df_meta.head()

In [ ]:
def parsear_linea_log(linea):
    patron = r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\w+)\s+([\w\.]+)\s+(\[.*?\])\s+(.+)$'
    match = re.match(patron, linea.strip())
    if match:
        return {
            'timestamp': match.group(1),
            'pid': int(match.group(2)),
            'level': match.group(3),
            'module': match.group(4),
            'request_id': match.group(5).strip('[]'),
            'message': match.group(6),
        }
    return None

def log_to_dataframe(ruta_archivo):
    lineas_parseadas = []
    with FS.open(ruta_archivo, "rt", encoding="utf-8", errors="ignore") as f:
        for linea in f:
            datos_linea = parsear_linea_log(linea)
            if datos_linea:
                lineas_parseadas.append(datos_linea)
    return pd.DataFrame(lineas_parseadas)

ruta_log = f"{DATASET_ROOT}/Nova/Test_1/logs/round_1/nova/nova-compute.log.bzip2.out"
df_logs = log_to_dataframe(ruta_log)
df_logs.head()

In [ ]:
def cargar_muestra_del_dataset(ruta_raiz, limite_tests=30):
    todos_los_logs = []
    componentes = ['Nova', 'Cinder', 'Neutron']

    for comp in componentes:
        ruta_comp = f"{ruta_raiz}/{comp}"
        if not FS.exists(ruta_comp):
            continue

        print(f"Procesando componente: {comp}...")
        carpetas_test = [p for p in FS.ls(ruta_comp, detail=False) if p.rstrip('/').split('/')[-1].startswith('Test_')][:limite_tests]
        print(f"   -> Encontrados {len(carpetas_test)} tests para procesar (Límite aplicado).")

        for carpeta_test in carpetas_test:
            test_id = carpeta_test.rstrip('/').split('/')[-1]
            for archivo_log in FS.glob(f"{carpeta_test}/**/*log*"):
                if archivo_log.endswith('/'):
                    continue

                round_id = next((part for part in archivo_log.rstrip('/').split('/') if part.startswith('round_')), 'no_round')
                lineas_parseadas = []
                with FS.open(archivo_log, "rt", encoding="utf-8", errors="ignore") as f:
                    for linea in f:
                        datos = parsear_linea_log(linea)
                        if datos:
                            datos['main_component'] = comp
                            datos['test_id'] = test_id
                            datos['round_id'] = round_id
                            datos['log_file_name'] = archivo_log.rstrip('/').split('/')[-1]
                            lineas_parseadas.append(datos)

                if lineas_parseadas:
                    todos_los_logs.append(pd.DataFrame(lineas_parseadas))

    if todos_los_logs:
        print("Unificando la muestra de logs")
        df_final = pd.concat(todos_los_logs, ignore_index=True)
        print(f"Muestra lista con {len(df_final)} líneas de log estructuradas.")
        return df_final

    return pd.DataFrame()

df_logs_muestra = cargar_muestra_del_dataset(DATASET_ROOT, limite_tests=30)
df_logs_muestra.head()